In [0]:
# NYC Taxi Pipeline Configuration

RAW_PATH = "/Volumes/workspace/nyc_taxi/raw_data/yellow_tripdata_2026-08.parquet"

PROCESSED_PATH = "/Volumes/workspace/nyc_taxi/raw_data/processed_taxi_data"

DAILY_METRICS_PATH = "/Volumes/workspace/nyc_taxi/raw_data/daily_taxi_metrics"

print("Pipeline configuration loaded!")

In [0]:
from pyspark.sql.functions import (
    col,
    unix_timestamp,
    to_date,
    hour
)

def clean_and_transform(df):

    # 1. Remove invalid records
    cleaned_df = df.filter(
        (col("fare_amount") >= 0) &
        (col("tpep_dropoff_datetime") >= col("tpep_pickup_datetime"))
    )

    # 2. Remove exact duplicates
    deduplicated_df = cleaned_df.dropDuplicates()

    # 3. Create additional columns
    transformed_df = (
        deduplicated_df
        .withColumn(
            "trip_duration_minutes",
            (
                unix_timestamp("tpep_dropoff_datetime") -
                unix_timestamp("tpep_pickup_datetime")
            ) / 60
        )
        .withColumn(
            "pickup_date",
            to_date(col("tpep_pickup_datetime"))
        )
        .withColumn(
            "pickup_hour",
            hour(col("tpep_pickup_datetime"))
        )
    )

    # 4. Remove trips with zero or negative duration
    valid_df = transformed_df.filter(
        col("trip_duration_minutes") > 0
    )

    return valid_df

In [0]:
from pyspark.sql.functions import col, count, avg, round, sum

def generate_daily_metrics(processed_df):

    # Select only August 2026 pickup dates
    august_df = processed_df.filter(
        (col("pickup_date") >= "2026-08-01") &
        (col("pickup_date") < "2026-09-01")
    )

    # Calculate daily metrics
    daily_metrics = (
        august_df
        .groupBy("pickup_date")
        .agg(
            count("*").alias("total_trips"),
            round(avg("fare_amount"), 2).alias("average_fare"),
            round(avg("trip_duration_minutes"), 2)
                .alias("average_duration_minutes"),
            round(sum("total_amount"), 2).alias("total_revenue")
        )
        .orderBy("pickup_date")
    )

    return daily_metrics

In [0]:
from pyspark.sql.functions import col

def validate_data(df):

    logger.info("Starting data quality checks...")

    # Check 1: Dataset should not be empty
    total_records = df.count()

    if total_records == 0:
        raise ValueError("Data quality failed: Dataset is empty!")

    # Check 2: No negative fares
    negative_fares = df.filter(col("fare_amount") < 0).count()

    if negative_fares > 0:
        raise ValueError(
            f"Data quality failed: {negative_fares} negative fares found!"
        )

    # Check 3: No invalid trip durations
    invalid_durations = df.filter(
        col("trip_duration_minutes") <= 0
    ).count()

    if invalid_durations > 0:
        raise ValueError(
            f"Data quality failed: {invalid_durations} invalid durations found!"
        )

    logger.info(f"Total records validated: {total_records}")
    logger.info("All data quality checks passed!")

    return True

In [0]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger("NYC_Taxi_Pipeline")

In [0]:
def run_nyc_taxi_pipeline():

    try:
        logger.info("Starting NYC Taxi ETL Pipeline...")

        # 1. Extract
        logger.info("Step 1: Reading raw data...")
        raw_df = spark.read.parquet(RAW_PATH)

        # 2. Transform
        logger.info("Step 2: Cleaning and transforming data...")
        processed_df = clean_and_transform(raw_df)
        # 3. validate
        logger.info("Step 3: Validating data quality...")
        validate_data(processed_df)

        # 4. Load processed data
        logger.info("Step 4: Saving processed data...")
        processed_df.write.mode("overwrite").parquet(PROCESSED_PATH)

        # 5. Generate daily metrics
        logger.info("Step 5: Generating daily metrics...")
        daily_metrics = generate_daily_metrics(processed_df)

        # 6. Load daily metrics
        logger.info("Step 6: Saving daily metrics...")
        daily_metrics.write.mode("overwrite").parquet(DAILY_METRICS_PATH)

        logger.info("NYC Taxi ETL Pipeline completed successfully!")

        return processed_df, daily_metrics

    except Exception:
        logger.exception("NYC Taxi ETL Pipeline failed!")
        raise

In [0]:
final_processed_df, final_daily_metrics = run_nyc_taxi_pipeline()